# LeukemiPrediag v2.1 — Probabilistic Clone Inference
Run uncertainty-aware candidate-clone inference on the real longitudinal feature table. Research use only.


In [7]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import os, sys

MYDRIVE = Path("/content/drive/MyDrive")

print("MyDrive exists:", MYDRIVE.exists())

# Find the project automatically
matches = list(MYDRIVE.glob("*/blood-cancer-early-detection"))

# Also check if it sits directly inside MyDrive
direct = MYDRIVE / "blood-cancer-early-detection"
if direct.exists():
    matches.append(direct)

print("\nFound:")
for p in matches:
    print(" -", p)

if not matches:
    raise FileNotFoundError(
        "Could not find blood-cancer-early-detection under MyDrive."
    )

PROJECT_ROOT = matches[0]
SRC = PROJECT_ROOT / "src"

os.chdir(PROJECT_ROOT)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("\nPROJECT_ROOT =", PROJECT_ROOT)
print("SRC =", SRC)
print("clone_inference exists =", (SRC / "clone_inference.py").exists())
print("real_data_adapter exists =", (SRC / "real_data_adapter.py").exists())

Mounted at /content/drive
MyDrive exists: True

Found:
 - /content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection

PROJECT_ROOT = /content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection
SRC = /content/drive/MyDrive/leukemia-prediagnose/blood-cancer-early-detection/src
clone_inference exists = True
real_data_adapter exists = True


In [8]:
from clone_inference import CloneInferenceConfig, run_clone_inference

print("✓ clone_inference imported")

✓ clone_inference imported


In [9]:
serial = pd.read_csv("data/processed/v2/real_serial_filtered.csv")
cfg = CloneInferenceConfig(n_draws=1000, pair_support_threshold=0.80)
result = run_clone_inference(serial, cfg)
result["audit"]


{'config': {'patient_col': 'patient_id',
  'cohort_col': 'cohort',
  'variant_col': 'variant_id',
  'time_col': 'time_years',
  'vaf_col': 'vaf',
  'depth_col': 'depth',
  'min_timepoints': 3,
  'n_draws': 1000,
  'seed': 17,
  'log_eps': 1e-05,
  'min_log_sigma': 0.05,
  'cohorts_without_call_depth': ('SardiNIA',),
  'corr_threshold': 0.8,
  'ratio_log_sd_threshold': 0.3,
  'slope_diff_threshold': 0.05,
  'pair_support_threshold': 0.8,
  'fast_growth_threshold': 0.05},
 'cohort_log_noise': {'GSE178936': 0.07423421328430369,
  'GSE288742': 0.09168606328141227,
  'SardiNIA': 0.12573507083174243},
 'n_patients': 265,
 'n_variants': 423,
 'n_pairs_evaluated': 238,
 'n_high_support_pairs': 21,
 'n_candidate_clones': 404,
 'n_multivariant_clones': 18,
 'n_patients_with_supported_structure': 18}

In [10]:
result["pair_support"].sort_values("support", ascending=False).head(20)


,shared_timepoints,support,p_corr,p_ratio_stable,p_slope_similar,corr_median,ratio_log_sd_median,slope_diff_median,patient_id,cohort,variant_a,variant_b,candidate_same_clone
208,5,0.988,0.995,0.908,0.992,0.957833,0.197236,0.013941,PD41190,SardiNIA,20:31022233.0:A>C,7:148504766.0:C>T,True
77,4,0.986,0.335,0.998,0.987,0.678031,0.131366,0.013727,CHIP_LBC36_021,GSE178936,chr2:25463181:C>T,chr2:25467198:G>T,True
209,5,0.986,0.994,0.924,0.992,0.954734,0.192663,0.012467,PD41190,SardiNIA,4:106190837.0:C>T,7:148504766.0:C>T,True
207,5,0.976,0.977,0.809,0.999,0.936334,0.231561,0.011238,PD41190,SardiNIA,20:31022233.0:A>C,4:106190837.0:C>T,True
222,4,0.970,0.977,0.917,0.981,0.960785,0.176567,0.014075,PD41262,SardiNIA,2:25463286.0:C>T,4:106158371.0:CACCA>C,True
235,3,0.964,1.000,0.920,0.961,0.997985,0.158260,0.016451,PD41346,SardiNIA,20:31022319.0:G>T,9:139400990.0:T>A,True
187,5,0.955,0.066,0.955,0.998,0.160755,0.180957,0.009783,PD41104,SardiNIA,1:1747229.0:T>C,2:25458661.0:T>C,True
195,3,0.948,0.988,0.929,0.814,0.987659,0.151816,0.025420,PD41122,SardiNIA,16:3779157.0:C>T,2:25469083.0:CC>C,True
216,3,0.940,0.503,0.939,0.991,0.801831,0.148281,0.013151,PD41242,SardiNIA,16:3900813.0:C>T,4:106193978.0:CTCCCTGG>C,True
227,5,0.933,1.000,0.729,0.933,0.987503,0.252258,0.026827,PD41276,SardiNIA,2:198267359.0:C>G,X:39922886.0:A>T,True


In [11]:
result["clone_posteriors"].query("n_variants > 1").sort_values("clone_p_fast_growth", ascending=False)


,patient_id,cohort,clone_id,n_variants,index_vaf_proxy,max_member_vaf,clone_growth_mean,clone_growth_median,clone_growth_sd,clone_growth_ci_low,clone_growth_ci_high,clone_p_growing,clone_p_fast_growth,clone_min_pair_support,clone_mean_pair_support
199,PD34527,SardiNIA,PD34527::clone_1,2,0.14820,0.1571,0.248551,0.248654,0.018088,0.213866,0.283758,1.000,1.000,0.874,0.874000
134,CHIP2024_396,GSE288742,CHIP2024_396::clone_1,2,0.19340,0.2012,0.153748,0.153466,0.013070,0.128217,0.178494,1.000,1.000,0.932,0.932000
356,PD41262,SardiNIA,PD41262::clone_1,2,0.17420,0.2089,0.100869,0.100560,0.010924,0.079002,0.122656,1.000,1.000,0.970,0.970000
358,PD41265,SardiNIA,PD41265::clone_1,2,0.05560,0.0584,0.078862,0.079172,0.007020,0.064417,0.092099,1.000,1.000,0.878,0.878000
396,PD41346,SardiNIA,PD41346::clone_1,2,0.27890,0.3152,0.231519,0.231783,0.011282,0.209404,0.253979,1.000,1.000,0.964,0.964000
364,PD41276,SardiNIA,PD41276::clone_1,2,0.35550,0.3591,0.170842,0.170243,0.008056,0.156259,0.187740,1.000,1.000,0.933,0.933000
306,PD41190,SardiNIA,PD41190::clone_1,3,0.29480,0.3460,0.106706,0.106741,0.006681,0.093451,0.119872,1.000,1.000,0.976,0.983333
268,PD41122,SardiNIA,PD41122::clone_1,2,0.03850,0.0389,0.152589,0.152121,0.017875,0.117998,0.187735,1.000,1.000,0.948,0.948000
258,PD41105,SardiNIA,PD41105::clone_1,2,0.06645,0.0804,0.073907,0.073627,0.010923,0.053826,0.096498,1.000,0.991,0.919,0.919000
185,PD34511,SardiNIA,PD34511::clone_1,2,0.07970,0.0910,0.067503,0.067706,0.012911,0.042118,0.092214,1.000,0.908,0.932,0.932000


## Important
`clone_p_growing` is probability that the clone growth slope is above zero under the measurement model. It is **not** probability of leukemia progression.
